## Python 1: Assignment 1 Part 2

In [1]:
import pandas as pd
import numpy as np

### Exercise 1

In [2]:
ba = pd.read_csv('StockData/BA.csv')
# print(ba.info())
# # NB y not Y since originsl fsts id 2-digit year not 4-digit year
# ba['Date'] = pd.to_datetime(ba['Date'], format='%m/%d/%y') # or format=r''
# ba.set_index(ba['Date'], inplace=True)
# print(ba.info())

# read data in again, setting index as we do so
ba = pd.read_csv('StockData/BA.csv', header=0, index_col='Date', parse_dates=True, date_format='%m/%d/%y')

# ba.info()

ba.describe()
ba.head()

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2013-10-01,117.970001,118.300003,117.080002,117.750000,102.874680,2880700
2013-10-02,117.379997,117.839996,116.279999,117.839996,102.953323,2871900
2013-10-03,117.349998,117.489998,114.730003,115.239998,100.681778,4455100
2013-10-04,115.269997,117.260002,115.250000,117.199997,102.394180,3998000
2013-10-07,115.510002,117.169998,115.400002,116.690002,101.948601,3822000


### Exercise 2

In [3]:
sp500 = pd.read_csv('StockData/SP500.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
ba = pd.read_csv('StockData/BA.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')

sp500oc = sp500[['Open','Close']].copy()
ba_oc = ba[['Open','Close']].copy()

sp500oc['Close_1'] = sp500oc['Close'].shift(1)
ba_oc['Close_1'] = ba_oc['Close'].shift(1)

sp500oc.head()
# ba_oc.head()

sp500oc['rtns'] = (sp500oc['Close'] - sp500oc['Close_1'])/sp500oc['Close_1']
sp500oc['log_rtns'] = np.log(sp500oc['Close']) - np.log(sp500oc['Close_1'])

ba_oc['rtns'] = (ba_oc['Close'] - ba_oc['Close_1'])/ba_oc['Close_1']
ba_oc['log_rtns'] = np.log(ba_oc['Close']) - np.log(ba_oc['Close_1'])

sp500oc.head()
ba_oc.head()

merged_df = sp500oc.merge(ba_oc, how='left', left_index=True, right_index=True, suffixes =('_sp','_ba'))
merged_df.head()

print('Mean SP500 Daily Return: {:%}'.format(np.mean(merged_df['rtns_sp'])))
print('Mean SP500 Daily Log-Return: {:%}'.format(np.mean(merged_df['log_rtns_sp'])))
print('Mean BA Daily Return: {:%}'.format(np.mean(merged_df['rtns_ba'])))
print('Mean BA Daily Log-Return: {:%}'.format(np.mean(merged_df['log_rtns_ba'])))

merged_df.describe()

Mean SP500 Daily Return: 0.046694%
Mean SP500 Daily Log-Return: 0.043670%
Mean BA Daily Return: 0.101280%
Mean BA Daily Log-Return: 0.091420%


,Open_sp,Close_sp,Close_1_sp,rtns_sp,log_rtns_sp,Open_ba,Close_ba,Close_1_ba,rtns_ba,log_rtns_ba
count,1260.000000,1260.000000,1259.000000,1259.000000,1259.000000,1259.000000,1259.000000,1258.000000,1258.000000,1258.000000
mean,2208.119022,2208.579183,2208.018896,0.000467,0.000437,181.936466,182.014718,181.863776,0.001013,0.000914
std,311.783587,311.699505,311.187978,0.007758,0.007771,77.847511,77.860511,77.706980,0.014015,0.014020
min,1656.989990,1655.449951,1655.449951,-0.040979,-0.041843,105.120003,108.440002,108.440002,-0.089290,-0.093531
25%,1984.504975,1985.515015,1985.489990,-0.002826,-0.002830,130.544998,130.590004,130.580005,-0.006296,-0.006316
50%,2101.550049,2101.265014,2101.040039,0.000489,0.000489,143.220001,143.029999,143.019997,0.001135,0.001134
75%,2434.364991,2434.777527,2434.145019,0.004560,0.004550,199.599998,199.799995,199.574997,0.009019,0.008979
max,2936.760010,2930.750000,2930.750000,0.039034,0.038291,372.000000,372.230011,372.230011,0.098795,0.094214


### Exercise 3

In [9]:
import pandas as pd

sp500 = pd.read_csv('StockData/SP500.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
ba = pd.read_csv('StockData/BA.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')

sp500oc = sp500[['Open','Close']]
ba_oc = ba[['Open','Close']]

# sp500oc['Close_1'] = sp500oc['Close'].shift(1)
# ba_oc['Close_1'] = ba_oc['Close'].shift(1)

sp500oc.head()
ba_oc.head()

# sp500oc['Pos_Day'] = sp500oc['Close'] > sp500oc['Open']
# ba_oc['Pos_Day'] = ba_oc['Close'] > ba_oc['Open']

merged_df = sp500oc.merge(ba_oc, how='left', left_index=True, right_index=True, suffixes =('_sp','_ba'))
merged_df.head()

,Open_sp,Close_sp,Open_ba,Close_ba
Date,,,,
2013-09-30,1687.260010,1681.550049,NaN,NaN
2013-10-01,1682.410034,1695.000000,117.970001,117.750000
2013-10-02,1691.900024,1693.869995,117.379997,117.839996
2013-10-03,1692.349976,1678.660034,117.349998,115.239998
2013-10-04,1678.790039,1690.500000,115.269997,117.199997


In [4]:
merged_df['ba_pos_sp_neg'] = merged_df.apply(
    lambda x: 1 if (x['Pos_Day_ba'] == True) and (x['Pos_Day_sp'] == False) else 0, axis=1)
merged_df.head(10)

merged_df.dropna(inplace=True)
merged_df['ba_pos_sp_neg'].value_counts()

ba_pos_sp_neg
0    1072
1     186
Name: count, dtype: int64

### Exercise 4

In [5]:
# Joining with a gap in the index
bbus = pd.read_csv('StockData/BB/BB_NYSE.csv', index_col='Date', parse_dates=True, date_format='%m/%d/%y')
bbus.head()
bbcad = pd.read_csv('StockData/BB/BB_TO.csv', index_col='Date', parse_dates=True, date_format='%m/%d/%y')
bbcad.head()

# Subset to only Jan
bbus = bbus.loc['2018-01']
bbus
bbcad = bbcad.loc['2018-01']
bbcad
bb = bbcad.join(bbus, lsuffix='_tse', rsuffix='_nyse')
bb.loc['2018-01-12':'2018-01-17']

FileNotFoundError: [Errno 2] No such file or directory: 'StockData/BB/BB_NYSE.csv'

### Exercise 5

In [ ]:
sp500 = pd.read_csv('StockData/SP500.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
sp500.columns = [i.lower().replace(' ','_') for i in sp500.columns]
sp500.head()

sp500['adj_close_1'] = sp500['adj_close'].shift()
sp500['log_rtn_c_to_o'] = np.log(sp500['open']) - np.log(sp500['adj_close_1'])
sp500['log_rtn_o_to_c'] = np.log(sp500['adj_close']) - np.log(sp500['open'])

# When finding daily vol, the numbers can become really small decimals.
# We can multiply the returns by 100 before squaring if a numerical problem occurs
sp500['vol_c_to_o'] = (sp500['log_rtn_c_to_o'] ** 2)
sp500['vol_o_to_c'] = (sp500['log_rtn_o_to_c'] ** 2)

vol_weight = 0.5

sp500['vol'] = sp500['vol_o_to_c'] * vol_weight + sp500['vol_c_to_o'] * (1-vol_weight)

sp500['vol_1'] = sp500['vol'].shift(1)
sp500['vol_ma_5'] = sp500['vol_1'].rolling(window = 5, min_periods=5).mean()
sp500['vol_ma_21'] = sp500['vol_1'].rolling(window = 21, min_periods=21).mean()

# save the dataFrame to a file
# NB make sure 'output' folder already exists
sp500.to_csv("Output/SP500ex5.csv")
sp500.to_excel("Output/SP500ex5.xlsx")


sp500.head(30)

### Exercise 6

In [ ]:
ba = pd.read_csv('StockData/BA.csv', header=0, index_col=0, parse_dates=True, date_format='%m/%d/%y')
ba.columns = [i.lower().replace(' ','_') for i in ba.columns]
ba.head()

financials = pd.read_csv('StockData/fundamentals.csv', index_col=0)
financials.columns = [i.lower().replace(' ','_') for i in financials.columns]
# financials.info()
# financials

# # Convert to month end to handle alignment with reporting date
# # using the previous quarter end to not have information early
ba['match_date'] = ba.index + pd.offsets.MonthEnd(-3)

financials['period_ending'] = pd.to_datetime(financials['period_ending'], format='%m/%d/%y')
financials['match_date'] = financials['period_ending'] + pd.offsets.MonthEnd(0)

financials['ROA'] = financials['net_income']/financials['total_assets']

max_date = max(financials[financials['ticker_symbol']=='BA']['match_date'])
min_date = min(financials[financials['ticker_symbol']=='BA']['match_date'])

ba_fin = ba.merge(financials[financials['ticker_symbol']=='BA'][['match_date','ROA']], how='left')
ba_fin.info()
ba_fin
ba_fin['ROA'] = ba_fin['ROA'].ffill()
ba_fin['ROA']

ba_fin = ba_fin[ba_fin['match_date'] <= (max_date + pd.DateOffset(years=1))]
ba_fin = ba_fin[ba_fin['match_date'] >= (min_date)]
ba_fin

### Exercise 7

In [ ]:
#Open FF3 - Fama French Three Factors and CAT
ba = pd.read_csv('StockData/ba.csv', header=0, index_col=0, parse_dates = True, date_format='%m/%d/%y')
ba.columns = [i.lower().replace(' ','_') for i in ba.columns]
ba.head()

# NB no dat parsing here
ff3 = pd.read_csv('StockData/FF3_monthly.csv', header=0, index_col=0)
# the dates in the FF3 file cannot be automatically parsed
# will be imported as a int64index

# # The index is not date time
ff3.info()

# # Create a temporary date column and copy the index
ff3['date'] = ff3.index

# # Parse the date time, default is beginning of the month, offset to the end of the month
ff3['date'] = pd.to_datetime(ff3.index,format='%Y%m') + pd.offsets.MonthEnd(0)
ff3
ff3.set_index(['date'], drop=True, inplace=True)
ff3.head()

# # Same rules as previously defined
ohlc_rule = {'open':'first', 'high':'max', 'low':'min',
            'close':'last', 'volume':'sum', 'adj_close':'last'}

# # Resample the daily data into monthly data using resample and agg
ba_mon = ba.resample('ME').agg(ohlc_rule)
# ba_mon.to_csv('output/ba_mon.csv')
ba_mon.head()

# # Log Returns on adjust close
ba_mon['log_rtns'] = np.log(ba_mon['adj_close']) - np.log(ba_mon['adj_close'].shift(1))

# # Merge on the index, no need for suffix because there are no duplicate columns
ba_ff = pd.merge(ba_mon, ff3, how='left', left_index=True, right_index=True)
ba_ff.head()
# # Notice how the returns are multiplied by 100 for the FF3 data, correct this for ba
ba_ff['log_rtns'] = ba_ff['log_rtns'] * 100
ba_ff.head()
# ba_ff.info()
# ba_ff.to_csv('output/BA_FF3_mon.csv')